In [11]:
import sklearn
import pandas as pd
print("Environment is healthy!")

Environment is healthy!


In [12]:
# notebooks/02_preprocessing.ipynb

import pandas as pd
import numpy as np
import os
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from imblearn.over_sampling import SMOTE

In [13]:
# Load raw data again (clean pipeline practice)
df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Clean column names
df.columns = df.columns.str.strip()

# Convert TotalCharges
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Drop customerID safely
df.drop(columns=['customerID'], errors='ignore', inplace=True)

df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [14]:
# Handle missing values
print(df.isnull().sum())

df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64


C:\Users\lenovo\AppData\Local\Temp\ipykernel_13852\871485826.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)


In [15]:
# Features & target
X = df.drop('Churn', axis=1)
y = df['Churn'].map({'Yes':1, 'No':0})

In [16]:
# Column types
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=np.number).columns.tolist()

In [17]:
scaler = StandardScaler()

X_train_res[numerical_cols] = scaler.fit_transform(X_train_res[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

NameError: name 'X_train_res' is not defined

In [ ]:
encoder = OneHotEncoder(drop='first', sparse_output=False)

X_encoded = pd.DataFrame(
    encoder.fit_transform(X[categorical_cols]),
    columns=encoder.get_feature_names_out(categorical_cols)
)

X_encoded.index = X.index

In [ ]:
X_final = pd.concat([X[numerical_cols], X_encoded], axis=1)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
print(X_train.isnull().sum().sort_values(ascending=False).head(10))

TotalCharges                      8
SeniorCitizen                     0
tenure                            0
MonthlyCharges                    0
gender_Male                       0
Partner_Yes                       0
Dependents_Yes                    0
PhoneService_Yes                  0
MultipleLines_No phone service    0
MultipleLines_Yes                 0
dtype: int64


In [ ]:
X_train[numerical_cols] = X_train[numerical_cols].fillna(X_train[numerical_cols].median())
X_test[numerical_cols] = X_test[numerical_cols].fillna(X_train[numerical_cols].median())


In [ ]:
print("NaNs in X_train:", X_train.isnull().sum().sum())
print("NaNs in X_test:", X_test.isnull().sum().sum())

NaNs in X_train: 0
NaNs in X_test: 0


In [ ]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

In [ ]:
print(y_train.value_counts())
print(y_train_res.value_counts())

Churn
0    4139
1    1495
Name: count, dtype: int64
Churn
0    4139
1    4139
Name: count, dtype: int64


In [ ]:
print("Final Training Shape:", X_train_res.shape)
print("Final Test Shape:", X_test.shape)

Final Training Shape: (8278, 30)
Final Test Shape: (1409, 30)


In [ ]:
import os, pickle

# Absolute safe path
base_path = os.path.abspath("..")

data_path = os.path.join(base_path, "data", "processed")
model_path = os.path.join(base_path, "models")

os.makedirs(data_path, exist_ok=True)
os.makedirs(model_path, exist_ok=True)

# Save datasets
X_train_res.to_csv(os.path.join(data_path, "X_train.csv"), index=False)
X_test.to_csv(os.path.join(data_path, "X_test.csv"), index=False)
y_train_res.to_csv(os.path.join(data_path, "y_train.csv"), index=False)
y_test.to_csv(os.path.join(data_path, "y_test.csv"), index=False)

# Save models
with open(os.path.join(model_path, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)

with open(os.path.join(model_path, "encoder.pkl"), "wb") as f:
    pickle.dump(encoder, f)

print("✅ Files saved successfully!")



NameError: name 'X_train_res' is not defined

In [ ]:
import os
print(os.getcwd())

f:\churnSense\churnsense\notebooks
